# Discover Anomalies - v10.2 Edition

Find **anomalous and interesting** translations in the v10.2 model.

## What We're Looking For:
1. **Semantic Mismatches** - Unexpected translations
2. **Polysemy** - Words with multiple distinct meanings
3. **Cultural Concepts** - Egyptian-specific terms → modern concepts
4. **Surprising Patterns** - Interesting linguistic connections

## Setup

In [ ]:
import json
import pickle
import numpy as np
from pathlib import Path
from gensim.models import FastText, KeyedVectors
from sklearn.linear_model import Ridge
from collections import defaultdict
from tqdm.auto import tqdm

# Configuration
REPO_ROOT = Path.cwd() if Path.cwd().name == 'heiroglyphy' else Path.cwd().parent
V10_ROOT = REPO_ROOT / "heiro_v10_refinement"

print("🔍 DISCOVER ANOMALIES - v10.2")
print("=" * 60)

## Load Models

In [ ]:
print("Loading models...")

# FastText
hier_model = FastText.load(str(REPO_ROOT / "heiro_v7_FastTextVisual/models/fasttext_v7.model"))
text_embeddings = hier_model.wv

# Visual embeddings
with open(REPO_ROOT / "heiro_v9_use_visuals_again/data/processed/visual_embeddings_768d.pkl", 'rb') as f:
    visual_embeddings = pickle.load(f)

# Lexicon
with open(V10_ROOT / "data/lexicon_trans_to_codes.json", 'r') as f:
    trans_to_codes = json.load(f)

# GloVe
print("Loading GloVe...")
english_embeddings = KeyedVectors.load_word2vec_format(
    str(REPO_ROOT / "heiro_v5_getdata/data/processed/glove.6B.300d.txt"),
    binary=False, no_header=True
)

# Anchors
with open(REPO_ROOT / "heiro_v6_BERT/data/processed/anchors.json", 'r') as f:
    anchors = json.load(f)

print(f"✓ Loaded all models")

## Create Fused Embeddings & Train Aligner

In [ ]:
print("Creating fused embeddings...")
fused_embeddings = {}

for word in tqdm(text_embeddings.index_to_key, desc="Fusing"):
    text_vec = text_embeddings[word]
    
    visual_vec = None
    if word in trans_to_codes:
        code_sequences = trans_to_codes[word]
        vectors = []
        for seq in code_sequences:
            seq_vectors = [visual_embeddings[code] for code in seq if code in visual_embeddings]
            if seq_vectors:
                vectors.append(np.mean(seq_vectors, axis=0))
        if vectors:
            visual_vec = np.mean(vectors, axis=0)
    
    if visual_vec is None:
        visual_vec = np.zeros(768)
    
    fused_embeddings[word] = np.concatenate([text_vec, visual_vec])

print(f"✓ Created {len(fused_embeddings)} embeddings")

In [ ]:
print("Training aligner...")
X, Y = [], []

for anchor in anchors:
    egy = anchor['hieroglyphic']
    eng = anchor['english'].lower()
    if egy in fused_embeddings and eng in english_embeddings:
        X.append(fused_embeddings[egy])
        Y.append(english_embeddings[eng])

X, Y = np.array(X), np.array(Y)
aligner = Ridge(alpha=1.0)
aligner.fit(X, Y)

print(f"✓ Trained on {len(X)} anchors (R²={aligner.score(X, Y):.3f})")

## Helper Functions

In [ ]:
def get_translations(egy_word, k=10):
    """Get top-k English translations."""
    if egy_word not in fused_embeddings:
        return []
    
    egy_vec = fused_embeddings[egy_word]
    eng_pred = aligner.predict([egy_vec])[0]
    neighbors = english_embeddings.similar_by_vector(eng_pred, topn=k)
    return [(word, sim) for word, sim in neighbors]

def get_expected_translation(egy_word):
    """Get the expected translation from anchors."""
    for anchor in anchors:
        if anchor['hieroglyphic'] == egy_word:
            return anchor['english'].lower()
    return None

## 1. Find Semantic Mismatches

Words where the top prediction is **very different** from the expected meaning.

In [ ]:
print("\n" + "=" * 60)
print("🔴 SEMANTIC MISMATCHES")
print("=" * 60)

mismatches = []

for anchor in anchors[:500]:  # Sample for speed
    egy = anchor['hieroglyphic']
    expected = anchor['english'].lower()
    
    if egy not in fused_embeddings or expected not in english_embeddings:
        continue
    
    translations = get_translations(egy, k=10)
    top_words = [w for w, s in translations]
    
    # Check if expected is NOT in top 10
    if expected not in top_words:
        mismatches.append({
            'egyptian': egy,
            'expected': expected,
            'predicted': top_words[0] if top_words else 'N/A',
            'top5': top_words[:5]
        })

print(f"\nFound {len(mismatches)} mismatches (expected not in top-10)\n")

# Show most interesting ones
for i, m in enumerate(mismatches[:15], 1):
    print(f"{i:2d}. {m['egyptian']}")
    print(f"    Expected: {m['expected']}")
    print(f"    Got:      {m['predicted']}")
    print(f"    Top-5:    {', '.join(m['top5'])}")
    print()

## 2. Find Polysemous Words

Egyptian words that map to **semantically diverse** English words.

In [ ]:
print("\n" + "=" * 60)
print("🟡 POLYSEMOUS WORDS (Multiple Meanings)")
print("=" * 60)

# Sample high-frequency words
sample_words = text_embeddings.index_to_key[:200]

polysemy_scores = []

for word in sample_words:
    if word not in fused_embeddings:
        continue
    
    translations = get_translations(word, k=10)
    
    # Calculate semantic diversity (variance in similarities)
    sims = [s for w, s in translations]
    if len(sims) >= 5:
        diversity = np.std(sims)  # Higher std = more diverse meanings
        polysemy_scores.append({
            'word': word,
            'diversity': diversity,
            'translations': [w for w, s in translations[:5]]
        })

# Sort by diversity
polysemy_scores.sort(key=lambda x: x['diversity'], reverse=True)

print(f"\nTop 15 most polysemous words:\n")
for i, item in enumerate(polysemy_scores[:15], 1):
    print(f"{i:2d}. {item['word']} (diversity={item['diversity']:.3f})")
    print(f"    → {', '.join(item['translations'])}")
    print()

## 3. Cultural Concept Mappings

Egyptian-specific terms and what modern concepts they map to.

In [ ]:
print("\n" + "=" * 60)
print("🟢 CULTURAL CONCEPTS")
print("=" * 60)

cultural_words = {
    "nṯr": "God/Divine",
    "nsw": "King",
    "mꜣꜥ.t": "Maat/Truth/Order",
    "kꜣ": "Ka/Life Force",
    "bꜣ": "Ba/Soul",
    "ꜥnḫ": "Life/Ankh",
    "ḏd": "Djed/Stability",
    "wꜣs": "Was/Power",
    "ḥtp": "Offering/Peace",
    "mꜣꜥ-ḫrw": "True of Voice",
    "Wsjr": "Osiris",
    "Ḥr,w": "Horus",
    "rʾ": "Re/Sun",
    "jnpw": "Anubis"
}

print("\nHow Egyptian concepts map to modern English:\n")

for egy, meaning in cultural_words.items():
    translations = get_translations(egy, k=8)
    if translations:
        print(f"📜 {egy} ({meaning})")
        for i, (word, sim) in enumerate(translations, 1):
            print(f"  {i}. {word} ({sim:.3f})")
        print()

## 4. Surprising High-Confidence Translations

Translations with **very high similarity** that might reveal interesting patterns.

In [ ]:
print("\n" + "=" * 60)
print("🔵 HIGH-CONFIDENCE SURPRISES")
print("=" * 60)

high_conf = []

for word in text_embeddings.index_to_key[:500]:
    if word not in fused_embeddings:
        continue
    
    translations = get_translations(word, k=1)
    if translations:
        eng_word, sim = translations[0]
        if sim > 0.7:  # Very high confidence
            high_conf.append({
                'egyptian': word,
                'english': eng_word,
                'similarity': sim
            })

high_conf.sort(key=lambda x: x['similarity'], reverse=True)

print(f"\nTop 20 highest-confidence translations:\n")
for i, item in enumerate(high_conf[:20], 1):
    print(f"{i:2d}. {item['egyptian']} → {item['english']} ({item['similarity']:.3f})")

## 5. Words with Visual Features

Check if words with visual mappings perform better.

In [ ]:
print("\n" + "=" * 60)
print("🖼️  VISUAL vs TEXT-ONLY PERFORMANCE")
print("=" * 60)

visual_words = []
text_only_words = []

for anchor in anchors[:200]:
    egy = anchor['hieroglyphic']
    expected = anchor['english'].lower()
    
    if egy not in fused_embeddings or expected not in english_embeddings:
        continue
    
    translations = get_translations(egy, k=10)
    top_words = [w for w, s in translations]
    
    # Check if correct
    correct = expected in top_words
    rank = top_words.index(expected) + 1 if correct else None
    
    if egy in trans_to_codes:
        visual_words.append({'word': egy, 'correct': correct, 'rank': rank})
    else:
        text_only_words.append({'word': egy, 'correct': correct, 'rank': rank})

visual_acc = sum(1 for w in visual_words if w['correct']) / len(visual_words) * 100 if visual_words else 0
text_acc = sum(1 for w in text_only_words if w['correct']) / len(text_only_words) * 100 if text_only_words else 0

print(f"\nWords with visual features: {len(visual_words)}")
print(f"  Top-10 Accuracy: {visual_acc:.1f}%")
print(f"\nWords without visual: {len(text_only_words)}")
print(f"  Top-10 Accuracy: {text_acc:.1f}%")
print(f"\nDifference: {visual_acc - text_acc:+.1f}%")

## Summary

In [ ]:
print("\n" + "=" * 60)
print("📊 SUMMARY")
print("=" * 60)
print(f"\n✓ Analysis complete!")
print(f"\nKey Findings:")
print(f"  • {len(mismatches)} semantic mismatches found")
print(f"  • {len(polysemy_scores)} polysemous words analyzed")
print(f"  • {len(high_conf)} high-confidence translations")
print(f"  • Visual features: {visual_acc:.1f}% vs Text-only: {text_acc:.1f}%")